In [1]:
import pandas as pd
import numpy as np
import os

BASE_PATH = ".."
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

TRANS_PATH = os.path.join(
    PROCESSED_PATH,
    "transactions.csv"
)

trans_df = pd.read_csv(TRANS_PATH)

# Create account IDs
trans_df["Sender Account ID"] = (
    trans_df["From Bank"].astype(str)
    + "_"
    + trans_df["Account"].astype(str)
)

trans_df["Receiver Account ID"] = (
    trans_df["To Bank"].astype(str)
    + "_"
    + trans_df["Account.1"].astype(str)
)

# Convert timestamp
trans_df["Timestamp"] = pd.to_datetime(
    trans_df["Timestamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

# Convert amount
trans_df["Amount Paid"] = pd.to_numeric(
    trans_df["Amount Paid"],
    errors="coerce"
).fillna(0)

# Keep original transaction order
trans_df["_Original Order"] = np.arange(len(trans_df))

# Sort chronologically by sender account
trans_df = trans_df.sort_values(
    ["Sender Account ID", "Timestamp", "_Original Order"]
).reset_index(drop=True)

print("Transactions:", len(trans_df))
print(trans_df[
    [
        "Transaction ID",
        "Sender Account ID",
        "Timestamp",
        "Amount Paid"
    ]
].head())

Transactions: 6924049
   Transaction ID Sender Account ID Timestamp  Amount Paid
0          502624       0_800060CE0       NaT     40126.40
1          548498       0_800060CE0       NaT       487.00
2          548499       0_800060CE0       NaT      3364.82
3          571022       0_800060CE0       NaT   8555553.06
4          571023       0_800060CE0       NaT   4565543.93


In [4]:
# CELL 2 — OPTIMIZED TEMPORAL FEATURES

# Remove rows where timestamp could not be parsed
trans_df = trans_df.dropna(subset=["Timestamp"]).copy()

# Sort chronologically within each sender account
trans_df = trans_df.sort_values(
    ["Sender Account ID", "Timestamp", "_Original Order"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 1. Time since previous transaction
# ---------------------------------------------------------

trans_df["Previous Timestamp"] = (
    trans_df.groupby("Sender Account ID")["Timestamp"].shift(1)
)

trans_df["Time Since Previous Transaction"] = (
    trans_df["Timestamp"] -
    trans_df["Previous Timestamp"]
).dt.total_seconds()

# ---------------------------------------------------------
# 2. Fast historical window calculations
# ---------------------------------------------------------

trans_df["Transactions Last 1 Hour"] = 0
trans_df["Transactions Last 24 Hours"] = 0
trans_df["Transactions Last 7 Days"] = 0
trans_df["Amount Last 24 Hours"] = 0.0


# Process one sender account at a time
for account_id, group_index in trans_df.groupby(
    "Sender Account ID",
    sort=False
).groups.items():

    idx = np.asarray(group_index)

    times = (
        trans_df.loc[idx, "Timestamp"]
        .astype("int64")
        .to_numpy()
    )

    amounts = (
        trans_df.loc[idx, "Amount Paid"]
        .to_numpy()
    )

    # Find where previous transactions begin
    one_hour = (
        times - np.int64(pd.Timedelta(hours=1).value)
    )

    twenty_four_hours = (
        times - np.int64(pd.Timedelta(hours=24).value)
    )

    seven_days = (
        times - np.int64(pd.Timedelta(days=7).value)
    )

    # Search only previous transactions
    left_1h = np.searchsorted(
        times,
        one_hour,
        side="left"
    )

    left_24h = np.searchsorted(
        times,
        twenty_four_hours,
        side="left"
    )

    left_7d = np.searchsorted(
        times,
        seven_days,
        side="left"
    )

    # Current transaction is excluded
    positions = np.arange(len(times))

    trans_df.loc[idx, "Transactions Last 1 Hour"] = (
        positions - left_1h
    )

    trans_df.loc[idx, "Transactions Last 24 Hours"] = (
        positions - left_24h
    )

    trans_df.loc[idx, "Transactions Last 7 Days"] = (
        positions - left_7d
    )

    # Prefix sum for fast 24-hour amount calculation
    cumulative_amount = np.concatenate(
        ([0], np.cumsum(amounts))
    )

    trans_df.loc[idx, "Amount Last 24 Hours"] = (
        cumulative_amount[positions]
        - cumulative_amount[left_24h]
    )

print("Temporal features calculated successfully.")

Temporal features calculated successfully.


In [5]:
# CELL 3 — CREATE TEMPORAL DATASET

temporal_features_df = trans_df[
    [
        "Transaction ID",
        "Sender Account ID",
        "Receiver Account ID",
        "Transactions Last 1 Hour",
        "Transactions Last 24 Hours",
        "Transactions Last 7 Days",
        "Amount Last 24 Hours",
        "Time Since Previous Transaction"
    ]
].copy()

# First transaction for an account has no previous transaction
temporal_features_df[
    "Time Since Previous Transaction"
] = (
    temporal_features_df[
        "Time Since Previous Transaction"
    ].fillna(0)
)

print("Temporal dataset created.")
print("\nShape:", temporal_features_df.shape)

print("\nFirst 10 rows:")
print(temporal_features_df.head(10))

Temporal dataset created.

Shape: (0, 8)

First 10 rows:
Empty DataFrame
Columns: [Transaction ID, Sender Account ID, Receiver Account ID, Transactions Last 1 Hour, Transactions Last 24 Hours, Transactions Last 7 Days, Amount Last 24 Hours, Time Since Previous Transaction]
Index: []


In [6]:
# CELL 4 — VALIDATE AND SAVE

# Restore original transaction order
original_order = trans_df[
    ["Transaction ID", "_Original Order"]
]

temporal_features_df = temporal_features_df.merge(
    original_order,
    on="Transaction ID",
    how="left"
)

temporal_features_df = (
    temporal_features_df
    .sort_values("_Original Order")
    .drop(columns="_Original Order")
    .reset_index(drop=True)
)

expected_columns = [
    "Transaction ID",
    "Sender Account ID",
    "Receiver Account ID",
    "Transactions Last 1 Hour",
    "Transactions Last 24 Hours",
    "Transactions Last 7 Days",
    "Amount Last 24 Hours",
    "Time Since Previous Transaction"
]

print("Shape:", temporal_features_df.shape)

print("\nColumns:")
print(temporal_features_df.columns.tolist())

print("\nMissing values:")
print(temporal_features_df.isnull().sum())

print(
    "\nDuplicate Transaction IDs:",
    temporal_features_df["Transaction ID"].duplicated().sum()
)

# Validation
assert temporal_features_df.columns.tolist() == expected_columns
assert temporal_features_df["Transaction ID"].duplicated().sum() == 0

# Save
OUTPUT_PATH = os.path.join(
    PROCESSED_PATH,
    "temporal_features.csv"
)

temporal_features_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved successfully:")
print(OUTPUT_PATH)

Shape: (0, 8)

Columns:
['Transaction ID', 'Sender Account ID', 'Receiver Account ID', 'Transactions Last 1 Hour', 'Transactions Last 24 Hours', 'Transactions Last 7 Days', 'Amount Last 24 Hours', 'Time Since Previous Transaction']

Missing values:
Transaction ID                     0
Sender Account ID                  0
Receiver Account ID                0
Transactions Last 1 Hour           0
Transactions Last 24 Hours         0
Transactions Last 7 Days           0
Amount Last 24 Hours               0
Time Since Previous Transaction    0
dtype: int64

Duplicate Transaction IDs: 0

Saved successfully:
..\data\processed\temporal_features.csv
